# Tarea 2 - Pregunta 3
## Limpieza y transformación de datos

### Objetivo

En este notebook se realizará el proceso de limpieza y transformación
de los datos de contratación correspondientes a la entidad INVIAS,
tomando como punto de partida los resultados obtenidos en la auditoría
inicial.

El proceso busca preparar los datos para las etapas posteriores de
análisis estadístico y generación de resultados, conservando las
observaciones que puedan representar señales de alerta para la pregunta
de negocio.

### Criterios generales de limpieza

Las transformaciones realizadas en este notebook estarán orientadas a:

- Corregir formatos y tipos de datos.
- Estandarizar variables categóricas y de texto cuando sea necesario.
- Tratar los valores faltantes de acuerdo con el significado de cada
  variable.
- Identificar y tratar registros duplicados cuando corresponda.
- Preparar las variables financieras y temporales para el análisis.
- Conservar los casos identificados durante la auditoría que puedan ser
  relevantes para el análisis de riesgo contractual.

Las decisiones de eliminación o transformación de registros serán
documentadas y justificadas para garantizar la trazabilidad del proceso.

In [50]:
import pandas as pd

# Cargar el archivo de datos de INVIAS
df = pd.read_csv("../../invias.csv")

# Verificar las dimensiones iniciales del conjunto de datos
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

Número de filas: 25605
Número de columnas: 89


/var/folders/bd/ygh4x6gj0jd4vfxdxdjck_s00000gn/T/ipykernel_1425/192952563.py:4: DtypeWarning: Columns (0: direccion_de_ejecucion_del_contrato) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../invias.csv")


In [51]:
# Revisar los tipos de datos presentes en la columna problemática

df["direccion_de_ejecucion_del_contrato"].map(type).value_counts()

direccion_de_ejecucion_del_contrato
<class 'float'>    25602
<class 'str'>          3
Name: count, dtype: int64

In [52]:
# Identificar los registros que contienen texto en la columna direccion_de_ejecucion_del_contrato

registros_texto = df[
    df["direccion_de_ejecucion_del_contrato"].apply(lambda x: isinstance(x, str))
]

registros_texto[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "direccion_de_ejecucion_del_contrato"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,direccion_de_ejecucion_del_contrato
10409,CO1.PCCNTR.7898538,1151-2025,terminado,No definido
24503,CO1.PCCNTR.7894143,1147-2025,Cerrado,No definido
24550,CO1.PCCNTR.7898079,1152-2025,terminado,No definido


### 2.1 Tratamiento de `direccion_de_ejecucion_del_contrato`

Durante la carga inicial se identificó una advertencia de tipos mixtos en
la variable `direccion_de_ejecucion_del_contrato`.

La revisión mostró que 25.602 registros corresponden a valores faltantes
(`NaN`) y solamente 3 registros contienen texto. Los tres registros
presentan el valor `"No definido"`.

Dado que la variable no proporciona información efectiva sobre la
dirección de ejecución del contrato y presenta un nivel de ausencia de
información prácticamente total, se decide excluirla del conjunto de
variables utilizado para el análisis.

Esta decisión no implica eliminar registros, sino únicamente retirar una
variable que no aporta información analítica suficiente.

In [53]:
# Eliminar la variable sin información analítica suficiente

df = df.drop(columns=["direccion_de_ejecucion_del_contrato"])

print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

Número de filas: 25605
Número de columnas: 88


### 2.2 Conversión de variables de fecha

Las variables `fecha_de_firma`, `fecha_de_inicio_del_contrato` y
`fecha_de_fin_del_contrato` contienen información temporal relevante
para el análisis contractual.

Estas variables serán convertidas al formato datetime de pandas para
facilitar posteriormente el análisis por año, duración y relaciones
temporales entre los contratos.

Los valores faltantes serán conservados como valores nulos y no serán
imputados en esta etapa, dado que la ausencia de una fecha puede tener
significado para determinados estados contractuales.

In [54]:
# Convertir las variables de fecha al formato datetime

columnas_fecha = [
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato"
]

for columna in columnas_fecha:
    df[columna] = pd.to_datetime(df[columna], errors="coerce")

# Verificar los tipos de datos resultantes
df[columnas_fecha].dtypes

fecha_de_firma                  datetime64[us]
fecha_de_inicio_del_contrato    datetime64[us]
fecha_de_fin_del_contrato       datetime64[us]
dtype: object

### 2.3 Validación de las variables de fecha

Después de convertir las variables temporales al formato datetime, se
verifica que la transformación haya conservado la cantidad de valores
válidos y faltantes identificados durante la auditoría inicial.

In [55]:
# Verificar valores válidos y faltantes después de la conversión

for columna in columnas_fecha:
    valores_validos = df[columna].notna().sum()
    valores_faltantes = df[columna].isna().sum()

    print(f"{columna}:")
    print(f"  Valores válidos: {valores_validos}")
    print(f"  Valores faltantes: {valores_faltantes}")
    print()

fecha_de_firma:
  Valores válidos: 17477
  Valores faltantes: 8128

fecha_de_inicio_del_contrato:
  Valores válidos: 17795
  Valores faltantes: 7810

fecha_de_fin_del_contrato:
  Valores válidos: 20074
  Valores faltantes: 5531



### 2.4 Validación de coherencia temporal

Una vez convertidas las variables temporales al formato datetime, se
verifica la consistencia cronológica de las fechas contractuales.

Se revisará si existen registros en los cuales:

- La fecha de inicio sea anterior a la fecha de firma.
- La fecha de finalización sea anterior a la fecha de inicio.

Los registros con fechas faltantes no serán considerados inconsistentes
en esta validación, dado que no existe información suficiente para
establecer una relación cronológica.

In [56]:
# Validar relaciones cronológicas entre las fechas del contrato

inicio_antes_firma = df[
    df["fecha_de_inicio_del_contrato"].notna() &
    df["fecha_de_firma"].notna() &
    (df["fecha_de_inicio_del_contrato"] < df["fecha_de_firma"])
]

fin_antes_inicio = df[
    df["fecha_de_fin_del_contrato"].notna() &
    df["fecha_de_inicio_del_contrato"].notna() &
    (df["fecha_de_fin_del_contrato"] < df["fecha_de_inicio_del_contrato"])
]

print("Inicio anterior a firma:", len(inicio_antes_firma))
print("Fin anterior a inicio:", len(fin_antes_inicio))

Inicio anterior a firma: 2959
Fin anterior a inicio: 1


### 2.5 Investigación de fechas de inicio anteriores a la firma

La validación temporal identificó 2.959 contratos cuya fecha de inicio
registrada es anterior a la fecha de firma.

Debido al número significativo de casos, no se asumirán automáticamente
como errores de calidad de datos. Se realizará una revisión descriptiva
por estado del contrato para determinar si el comportamiento se concentra
en determinados estados contractuales.

Los registros serán conservados mientras se determina su tratamiento.

In [57]:
# Distribución por estado de los contratos con inicio anterior a la firma

inicio_antes_firma["estado_contrato"].value_counts()

estado_contrato
Cerrado       1156
terminado      826
Modificado     504
Aprobado       470
Suspendido       3
Name: count, dtype: int64

In [58]:
# Porcentaje de casos con inicio anterior a la firma sobre los contratos
# que tienen disponibles ambas fechas

total_con_firma_inicio = (
    df["fecha_de_firma"].notna() &
    df["fecha_de_inicio_del_contrato"].notna()
).sum()

porcentaje_inicio_antes_firma = (
    len(inicio_antes_firma) / total_con_firma_inicio * 100
)

print("Contratos con firma e inicio disponibles:", total_con_firma_inicio)
print("Inicio anterior a firma:", len(inicio_antes_firma))
print(f"Porcentaje: {porcentaje_inicio_antes_firma:.2f}%")

Contratos con firma e inicio disponibles: 17285
Inicio anterior a firma: 2959
Porcentaje: 17.12%


In [59]:
fin_antes_inicio[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "fecha_de_firma",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato
13181,CO1.PCCNTR.8746765,2320-2025,En ejecución,2025-12-31,2026-01-05,2025-09-04


### 2.6 Creación de indicadores de inconsistencias temporales

La revisión de las fechas identificó 2.959 contratos cuya fecha de inicio
es anterior a la fecha de firma, equivalentes al 17,12 % de los contratos
con ambas fechas disponibles.

Debido a la concentración de estos casos en determinados estados
contractuales, no se consideran automáticamente errores de digitación y
se conservan las fechas originales.

Adicionalmente, se identificó un contrato cuya fecha de finalización es
anterior a la fecha de inicio. Este caso será igualmente conservado.

Para preservar la información original y facilitar el análisis posterior,
se crean variables indicadoras que permitan identificar estas
inconsistencias sin modificar las fechas de origen.

In [60]:
# Crear indicadores de inconsistencias temporales

df["inicio_antes_firma"] = (
    df["fecha_de_inicio_del_contrato"].notna() &
    df["fecha_de_firma"].notna() &
    (df["fecha_de_inicio_del_contrato"] < df["fecha_de_firma"])
)

df["fin_antes_inicio"] = (
    df["fecha_de_fin_del_contrato"].notna() &
    df["fecha_de_inicio_del_contrato"].notna() &
    (df["fecha_de_fin_del_contrato"] < df["fecha_de_inicio_del_contrato"])
)

print("Inicio anterior a firma:", df["inicio_antes_firma"].sum())
print("Fin anterior a inicio:", df["fin_antes_inicio"].sum())

Inicio anterior a firma: 2959
Fin anterior a inicio: 1


### 2.7 Revisión de valores faltantes

Después de las transformaciones realizadas, se revisa nuevamente la
completitud de las variables.

Los valores faltantes no serán eliminados de manera general, dado que
pueden representar situaciones diferentes dependiendo de la variable y
del contexto contractual.

La decisión sobre su tratamiento se realizará variable por variable,
considerando su relevancia para el análisis y el porcentaje de
información disponible.

In [61]:
# Revisar valores faltantes después de las transformaciones

faltantes = pd.DataFrame({
    "no_nulos": df.notna().sum(),
    "faltantes": df.isna().sum(),
    "porcentaje_faltantes": df.isna().mean() * 100
})

faltantes = faltantes.sort_values(
    "porcentaje_faltantes",
    ascending=False
)

faltantes

,no_nulos,faltantes,porcentaje_faltantes
fecha_de_notificacion_de_prorrogacion,2726,22879,89.353642
fecha_fin_liquidacion,6036,19569,76.426479
fecha_inicio_liquidacion,6036,19569,76.426479
ultima_actualizacion,14553,11052,43.163445
fecha_de_firma,17477,8128,31.743800
...,...,...,...
proceso_de_compra,25605,0,0.000000
id_contrato,25605,0,0.000000
referencia_del_contrato,25605,0,0.000000
estado_contrato,25605,0,0.000000


### 2.8 Revisión de variables financieras

Debido al enfoque financiero de la Pregunta 3, se revisan de manera
particular las variables asociadas con el valor y la ejecución económica
de los contratos.

El análisis se concentra en las variables:

- `valor_del_contrato`
- `valor_facturado`
- `valor_pagado`
- `valor_pendiente_de_pago`
- `valor_pendiente_de_ejecucion`
- `valor_amortizado`
- `saldo_cdp`
- `saldo_vigencia`

Estas variables fueron previamente identificadas como disponibles para
el subconjunto financiero de 20.718 registros.

Se verificará nuevamente su completitud antes de continuar con las
transformaciones.

In [62]:
# Revisar completitud de las variables financieras

variables_financieras = [
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion",
    "valor_amortizado",
    "saldo_cdp",
    "saldo_vigencia"
]

faltantes_financieros = pd.DataFrame({
    "no_nulos": df[variables_financieras].notna().sum(),
    "faltantes": df[variables_financieras].isna().sum(),
    "porcentaje_faltantes": (
        df[variables_financieras].isna().mean() * 100
    )
})

faltantes_financieros

,no_nulos,faltantes,porcentaje_faltantes
valor_del_contrato,20718,4887,19.086116
valor_facturado,20718,4887,19.086116
valor_pagado,20718,4887,19.086116
valor_pendiente_de_pago,20718,4887,19.086116
valor_pendiente_de_ejecucion,20718,4887,19.086116
valor_amortizado,20718,4887,19.086116
saldo_cdp,20718,4887,19.086116
saldo_vigencia,20718,4887,19.086116


### 2.9 Construcción del subconjunto financiero

La revisión de completitud mostró que las ocho variables financieras
analizadas presentan exactamente el mismo patrón de faltantes: 20.718
registros con información y 4.887 registros sin información financiera.

Debido a que estos faltantes se presentan de manera conjunta, se
construye un subconjunto financiero a partir de los contratos que cuentan
con `valor_del_contrato`.

El conjunto de datos principal conserva los 25.605 contratos originales,
mientras que `df_financiero` contiene únicamente los 20.718 contratos con
información financiera disponible.

Los 4.887 registros sin información financiera no se eliminan del
conjunto principal, ya que podrían ser relevantes para otros análisis.

In [63]:
# Crear subconjunto de contratos con información financiera

df_financiero = df[
    df["valor_del_contrato"].notna()
].copy()

print("Registros del conjunto principal:", len(df))
print("Registros del subconjunto financiero:", len(df_financiero))

Registros del conjunto principal: 25605
Registros del subconjunto financiero: 20718


### 2.10 Revisión de variables categóricas

Se revisan las variables categóricas del conjunto de datos para
identificar valores inconsistentes, marcadores de información ausente,
errores de estandarización o valores atípicos en su representación.

No se modificarán los valores en esta etapa. Primero se identificarán
los valores con baja frecuencia y aquellos que puedan representar
información faltante o registros no estandarizados.

In [64]:
# Identificar las variables categóricas del conjunto de datos

columnas_categoricas = df.select_dtypes(
    include=["object", "string"]
).columns

print("Número de variables categóricas:", len(columnas_categoricas))
print("\nVariables categóricas:")
print(columnas_categoricas.tolist())

Número de variables categóricas: 65

Variables categóricas:
['id', 'version', 'created_at', 'updated_at', 'nombre_entidad', 'departamento', 'ciudad', 'localizacion', 'orden', 'sector', 'rama', 'entidad_centralizada', 'proceso_de_compra', 'id_contrato', 'referencia_del_contrato', 'estado_contrato', 'codigo_de_categoria_principal', 'descripcion_del_proceso', 'tipo_de_contrato', 'modalidad_de_contratacion', 'justificacion_modalidad_de', 'condiciones_de_entrega', 'tipodocproveedor', 'documento_proveedor', 'proveedor_adjudicado', 'es_grupo', 'es_pyme', 'habilita_pago_adelantado', 'liquidacion', 'obligacion_ambiental', 'obligaciones_postconsumo', 'reversion', 'origen_de_los_recursos', 'destino_gasto', 'espostconflicto', 'puntos_del_acuerdo', 'pilares_del_acuerdo', 'urlproceso', 'nombre_representante_legal', 'nacionalidad_representante_legal', 'domicilio_representante_legal', 'tipo_de_identificaci_n_representante_legal', 'identificaci_n_representante_legal', 'genero_representante_legal', 'ult

### 2.11 Revisión de categorías relevantes

Se revisan inicialmente las variables categóricas con mayor relevancia
para la caracterización de los contratos: estado, tipo y modalidad de
contratación, ubicación geográfica y proveedor adjudicado.

El objetivo es identificar categorías inconsistentes, valores que
representen ausencia de información y diferencias de estandarización.

En esta etapa no se modifican los valores originales.

In [65]:
# Revisar las categorías de las variables seleccionadas

variables_categoricas_relevantes = [
    "estado_contrato",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "departamento",
    "proveedor_adjudicado"
]

for columna in variables_categoricas_relevantes:
    print(f"\n--- {columna} ---")
    print("Número de categorías:", df[columna].nunique(dropna=False))
    print(df[columna].value_counts(dropna=False).head(15))


--- estado_contrato ---
Número de categorías: 11
estado_contrato
Cerrado              9263
Modificado           5434
terminado            3416
En ejecución         2512
Borrador             2438
Aprobado              872
Cancelado             629
Suspendido            361
enviado Proveedor     301
En aprobación         287
cedido                 92
Name: count, dtype: int64

--- tipo_de_contrato ---
Número de categorías: 16
tipo_de_contrato
Prestación de servicios       12058
NaN                            4887
Obra                           3409
Otro                           2326
Interventoría                  1530
Suministros                     593
Consultoría                     573
Compraventa                      81
Comodato                         74
Seguros                          32
Arrendamiento de inmuebles       31
Acuerdo Marco de Precios          5
Concesión                         2
Decreto 092 de 2017               2
Arrendamiento de muebles          1
Name: count, d

In [66]:
# Revisar variabilidad de la ubicación geográfica

print("Ciudades:")
print(df["ciudad"].value_counts(dropna=False))

print("\nNúmero de ciudades:", df["ciudad"].nunique(dropna=False))

Ciudades:
ciudad
Bogotá    25605
Name: count, dtype: int64

Número de ciudades: 1


### 2.12 Eliminación de variable sin variabilidad

La variable `departamento` presenta una única categoría para la totalidad
de los 25.605 registros: `Distrito Capital de Bogotá`.

Al no presentar variabilidad, esta variable no aporta información
discriminante para el análisis y se elimina del conjunto de datos.

In [67]:
# Eliminar variable sin variabilidad

df = df.drop(columns=["departamento"])

print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Número de filas: 25605
Número de columnas: 89


In [68]:
print("departamento presente:", "departamento" in df.columns)

departamento presente: False


### 2.13 Revisión de valores especiales en la referencia del contrato

Se revisan valores de baja frecuencia presentes en `referencia_del_contrato`
para identificar posibles marcadores de ausencia de información o valores
no estandarizados.

No se modificarán los registros hasta determinar el tratamiento adecuado.

In [69]:
# Revisar valores especiales en referencia del contrato

valores_especiales = [".", "-", "0", "XXXX", "CANCELADO"]

for valor in valores_especiales:
    print(f"\nValor: {valor}")
    print("Cantidad:", (df["referencia_del_contrato"] == valor).sum())


Valor: .
Cantidad: 12

Valor: -
Cantidad: 6

Valor: 0
Cantidad: 5

Valor: XXXX
Cantidad: 4

Valor: CANCELADO
Cantidad: 7


In [70]:
# Revisar el contexto de los valores especiales

df[df["referencia_del_contrato"].isin(valores_especiales)][[
    "id_contrato",
    "referencia_del_contrato",
    "estado_contrato",
    "proveedor_adjudicado"
]].sort_values("referencia_del_contrato")

,id_contrato,referencia_del_contrato,estado_contrato,proveedor_adjudicado
2230,CO1.PCCNTR.3175096,-,Borrador,JUAN CARLOS VANEGAS MUÑOZ
5660,CO1.PCCNTR.3174995,-,Borrador,CARLOS JULIO ROMERO ANTURY
9167,CO1.PCCNTR.3182314,-,Borrador,JOSE DAVID PADILLA ROMERO
9280,CO1.PCCNTR.9258475,-,Borrador,NaN
11690,CO1.PCCNTR.3174994,-,Borrador,DANIEL GEOVANNY FONSECA ZAMBRANO
25441,CO1.PCCNTR.3175173,-,Borrador,Julio Andres Ulloa Palomo
52,CO1.PCCNTR.7398046,.,Cancelado,Sin Descripcion
2417,CO1.PCCNTR.7335390,.,Borrador,NaN
4247,CO1.PCCNTR.8974804,.,Borrador,NaN
4684,CO1.PCCNTR.8877841,.,Borrador,NaN


#### Tratamiento de valores especiales

La revisión contextual permitió identificar dos grupos de valores
especiales.

Los valores `.`, `-`, `0` y `XXXX` se consideran valores no informativos
para la variable `referencia_del_contrato`, debido a su baja frecuencia y
a que aparecen principalmente en contratos en estado Borrador o Cancelado.

Estos valores serán reemplazados por valores nulos (`NaN`), preservando la
distinción entre ausencia de información y una referencia contractual
válida.

El valor `CANCELADO` se conserva, debido a que se encuentra asociado a
registros cuyo estado contractual puede ser Cancelado y no existe
evidencia suficiente para considerarlo un error de calidad.

In [71]:
# Reemplazar valores no informativos en la referencia del contrato

valores_no_informativos = [".", "-", "0", "XXXX"]

df["referencia_del_contrato"] = df["referencia_del_contrato"].replace(
    valores_no_informativos,
    pd.NA
)

In [72]:
# Validar el tratamiento de valores especiales

for valor in valores_no_informativos:
    print(
        f"{valor}:",
        (df["referencia_del_contrato"] == valor).sum()
    )

print(
    "\nCANCELADO:",
    (df["referencia_del_contrato"] == "CANCELADO").sum()
)

print(
    "\nValores faltantes en referencia:",
    df["referencia_del_contrato"].isna().sum()
)

.: 0
-: 0
0: 0
XXXX: 0

CANCELADO: 7

Valores faltantes en referencia: 27


In [73]:
print("Valores faltantes en referencia:", 
      df["referencia_del_contrato"].isna().sum())

print("Total de registros:", len(df))

print("Referencias no faltantes:", 
      df["referencia_del_contrato"].notna().sum())

Valores faltantes en referencia: 27
Total de registros: 25605
Referencias no faltantes: 25578


In [74]:
print(df["referencia_del_contrato"].value_counts(dropna=False).head(10))

referencia_del_contrato
NaN             27
CANCELADO        7
3655-2023        6
0079-2022        4
4616-2023        4
4239-2023        4
0385-2025        4
4528-2023        4
1941 DE 2024     4
4134-2023        4
Name: count, dtype: int64


Se identificaron 27 valores no informativos (`.`, `-`, `0` y `XXXX`) en `referencia_del_contrato`, los cuales fueron transformados a valores faltantes. El valor `CANCELADO`, presente en 7 registros, fue conservado debido a que corresponde a una categoría observada en el contexto contractual y no se encontró evidencia suficiente para considerarlo un error.

### 2.14 Revisión de valores no informativos en proveedor adjudicado

La variable `proveedor_adjudicado` presenta 4.887 valores faltantes y
791 registros con el texto `Sin Descripcion`.

Se revisará el contexto de estos registros para determinar si `Sin
Descripcion` representa ausencia de información y debe homologarse con
los valores faltantes.

In [75]:
# Revisar el contexto de "Sin Descripcion"

df[df["proveedor_adjudicado"] == "Sin Descripcion"][[
    "id_contrato",
    "referencia_del_contrato",
    "estado_contrato",
    "proveedor_adjudicado",
    "valor_del_contrato"
]].head(20)

,id_contrato,referencia_del_contrato,estado_contrato,proveedor_adjudicado,valor_del_contrato
4,CO1.PCCNTR.3516681,CO1.PCCNTR.3516681,Cancelado,Sin Descripcion,0.0
18,CO1.PCCNTR.6838297,CO1.PCCNTR.6838297,Cancelado,Sin Descripcion,0.0
51,CO1.PCCNTR.8277719,CO1.PCCNTR.8277719,Borrador,Sin Descripcion,0.0
52,CO1.PCCNTR.7398046,NaN,Cancelado,Sin Descripcion,0.0
59,CO1.PCCNTR.7766838,CO1.PCCNTR.7766838,Cancelado,Sin Descripcion,0.0
84,CO1.PCCNTR.8813998,CO1.PCCNTR.8813998,Borrador,Sin Descripcion,0.0
120,CO1.PCCNTR.5353027,CO1.PCCNTR.5353027,Cancelado,Sin Descripcion,0.0
142,CO1.PCCNTR.344809,CO1.PCCNTR.344809,Borrador,Sin Descripcion,0.0
157,CO1.PCCNTR.5035123,CO1.PCCNTR.5035123,Borrador,Sin Descripcion,0.0
195,CO1.PCCNTR.484706,CO1.PCCNTR.484706,Cancelado,Sin Descripcion,0.0


In [76]:
# Contexto de los registros con proveedor sin descripción

sin_descripcion = df[df["proveedor_adjudicado"] == "Sin Descripcion"]

print("Total 'Sin Descripcion':", len(sin_descripcion))

print("\nEstados del contrato:")
print(sin_descripcion["estado_contrato"].value_counts())

print("\nValor del contrato:")
print(sin_descripcion["valor_del_contrato"].describe())

print("\nCon valor del contrato igual a 0:")
print(
    (sin_descripcion["valor_del_contrato"] == 0).sum()
)

print("\nCon referencia del contrato faltante:")
print(
    sin_descripcion["referencia_del_contrato"].isna().sum()
)

Total 'Sin Descripcion': 791

Estados del contrato:
estado_contrato
Borrador     524
Cancelado    267
Name: count, dtype: int64

Valor del contrato:
count    7.910000e+02
mean     9.190316e+11
std      1.867711e+13
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.410000e+14
Name: valor_del_contrato, dtype: float64

Con valor del contrato igual a 0:
774

Con referencia del contrato faltante:
2


#### Tratamiento de "Sin Descripcion"

Se identificaron 791 registros con el valor `Sin Descripcion` en la variable
`proveedor_adjudicado`.

Los registros corresponden exclusivamente a contratos en estado `Borrador`
(524) o `Cancelado` (267). Adicionalmente, 774 de los 791 registros presentan
un valor del contrato igual a cero.

Debido a este patrón, se considera que `Sin Descripcion` representa ausencia
de información sobre el proveedor adjudicado y se homologa con un valor
faltante (`NaN`).

Esta transformación no implica afirmar que los registros sean inválidos,
sino que se estandariza la representación de ausencia de información para
facilitar el análisis posterior.

In [77]:
# Homologar "Sin Descripcion" como valor faltante

df["proveedor_adjudicado"] = df["proveedor_adjudicado"].replace(
    "Sin Descripcion",
    pd.NA
)

In [78]:
# Validar la transformación

print(
    "Registros con 'Sin Descripcion':",
    (df["proveedor_adjudicado"] == "Sin Descripcion").sum()
)

print(
    "Valores faltantes en proveedor_adjudicado:",
    df["proveedor_adjudicado"].isna().sum()
)

Registros con 'Sin Descripcion': 0
Valores faltantes en proveedor_adjudicado: 5678


In [ ]:
# Revisar valores faltantes después de las transformaciones

faltantes = pd.DataFrame({
    "no_nulos": df.notna().sum(),
    "faltantes": df.isna().sum(),
})

faltantes["porcentaje_faltantes"] = (
    faltantes["faltantes"] / len(df) * 100
)

faltantes.sort_values(
    "porcentaje_faltantes",
    ascending=False
).head(20)

,no_nulos,faltantes,porcentaje_faltantes
fecha_de_notificacion_de_prorrogacion,2726,22879,89.353642
fecha_inicio_liquidacion,6036,19569,76.426479
fecha_fin_liquidacion,6036,19569,76.426479
ultima_actualizacion,14553,11052,43.163445
fecha_de_firma,17477,8128,31.743800
fecha_de_inicio_del_contrato,17795,7810,30.501855
proveedor_adjudicado,19927,5678,22.175356
fecha_de_fin_del_contrato,20074,5531,21.601250
nombre_ordenador_del_gasto,20699,4906,19.160320
numero_de_cuenta,20699,4906,19.160320


In [80]:
# Identificar el patrón común de faltantes

columnas_4906 = [
    "nombre_ordenador_del_gasto",
    "numero_de_cuenta",
    "tipo_de_cuenta",
    "el_contrato_puede_ser_prorrogado",
    "objeto_del_contrato",
    "nombre_del_banco"
]

df[df[columnas_4906].isna().all(axis=1)][[
    "id_contrato",
    "referencia_del_contrato",
    "estado_contrato",
    "valor_del_contrato",
    "proveedor_adjudicado"
]].head(20)

,id_contrato,referencia_del_contrato,estado_contrato,valor_del_contrato,proveedor_adjudicado
5,CO1.PCCNTR.9279238,1150-2026,Aprobado,NaN,NaN
6,CO1.PCCNTR.3241665,303-2022,Cerrado,NaN,NaN
9,CO1.PCCNTR.9272734,1141-2026,En ejecución,NaN,NaN
14,CO1.PCCNTR.3012807,1773-2021,Modificado,NaN,NaN
27,CO1.PCCNTR.8747985,2213-2025.,En ejecución,NaN,NaN
28,CO1.PCCNTR.5273883,2920-2023,terminado,NaN,NaN
31,CO1.PCCNTR.2192385,509-2021,Cerrado,NaN,NaN
37,CO1.PCCNTR.9272917,1144-2026,Borrador,NaN,NaN
38,CO1.PCCNTR.6561364,2639-2024,En ejecución,NaN,NaN
39,CO1.PCCNTR.9261166,1058-2026,Modificado,NaN,NaN


In [82]:
# Comparar los registros con faltantes en el bloque general frente al subconjunto sin información financiera

columnas_4906 = [
    "nombre_ordenador_del_gasto",
    "numero_de_cuenta",
    "tipo_de_cuenta",
    "el_contrato_puede_ser_prorrogado",
    "objeto_del_contrato",
    "nombre_del_banco"
]

sin_bloque_general = df[columnas_4906].isna().all(axis=1)

columnas_financieras = [
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion",
    "valor_amortizado",
    "saldo_cdp",
    "saldo_vigencia"
]

sin_financiera = df[columnas_financieras].isna().all(axis=1)

print("Registros con bloque general faltante:", sin_bloque_general.sum())
print("Registros con bloque financiero faltante:", sin_financiera.sum())

print(
    "Registros presentes en ambos subconjuntos:",
    (sin_bloque_general & sin_financiera).sum()
)

print(
    "Registros con bloque general faltante pero información financiera:",
    (sin_bloque_general & ~sin_financiera).sum()
)

print(
    "Registros con información general pero bloque financiero faltante:",
    (~sin_bloque_general & sin_financiera).sum()
)

Registros con bloque general faltante: 4906
Registros con bloque financiero faltante: 4887
Registros presentes en ambos subconjuntos: 4887
Registros con bloque general faltante pero información financiera: 19
Registros con información general pero bloque financiero faltante: 0


In [83]:
# Estados de los registros con bloque general completamente faltante

print(
    df.loc[sin_bloque_general, "estado_contrato"]
      .value_counts(dropna=False)
)

estado_contrato
Cerrado              1653
Modificado           1306
En ejecución          794
terminado             470
Borrador              308
Suspendido            139
Aprobado               99
Cancelado              51
enviado Proveedor      51
cedido                 31
En aprobación           4
Name: count, dtype: int64


In [84]:
# Características básicas del subconjunto

print("Registros:", sin_bloque_general.sum())

print(
    "\nFecha de firma disponibles:",
    df.loc[sin_bloque_general, "fecha_de_firma"].notna().sum()
)

print(
    "Fecha de inicio disponibles:",
    df.loc[sin_bloque_general, "fecha_de_inicio_del_contrato"].notna().sum()
)

print(
    "Fecha de fin disponibles:",
    df.loc[sin_bloque_general, "fecha_de_fin_del_contrato"].notna().sum()
)

Registros: 4906

Fecha de firma disponibles: 18
Fecha de inicio disponibles: 18
Fecha de fin disponibles: 19


In [85]:
# Analizar el tipo de contrato en el subconjunto con faltantes estructurales

print(
    df.loc[sin_bloque_general, "tipo_de_contrato"]
      .value_counts(dropna=False)
)

tipo_de_contrato
NaN                        4887
Prestación de servicios      17
Interventoría                 2
Name: count, dtype: int64


In [86]:
# Comparar tipo de contrato entre el subconjunto con faltantes y el conjunto completo

print("Subconjunto con faltantes estructurales:")
print(
    df.loc[sin_bloque_general, "tipo_de_contrato"]
      .value_counts(normalize=True, dropna=False)
      .head(10)
)

print("\nConjunto completo:")
print(
    df["tipo_de_contrato"]
      .value_counts(normalize=True, dropna=False)
      .head(10)
)

Subconjunto con faltantes estructurales:
tipo_de_contrato
NaN                        0.996127
Prestación de servicios    0.003465
Interventoría              0.000408
Name: proportion, dtype: float64

Conjunto completo:
tipo_de_contrato
Prestación de servicios    0.470924
NaN                        0.190861
Obra                       0.133138
Otro                       0.090842
Interventoría              0.059754
Suministros                0.023160
Consultoría                0.022378
Compraventa                0.003163
Comodato                   0.002890
Seguros                    0.001250
Name: proportion, dtype: float64


In [87]:
# Verificar si los subconjuntos coinciden exactamente

ids_sin_bloque_general = set(
    df.loc[sin_bloque_general, "id_contrato"]
)

ids_sin_financiera = set(
    df.loc[sin_financiera, "id_contrato"]
)

print(
    "IDs presentes en ambos subconjuntos:",
    len(ids_sin_bloque_general & ids_sin_financiera)
)

print(
    "IDs solo en bloque general:",
    len(ids_sin_bloque_general - ids_sin_financiera)
)

print(
    "IDs solo en bloque financiero:",
    len(ids_sin_financiera - ids_sin_bloque_general)
)

IDs presentes en ambos subconjuntos: 4887
IDs solo en bloque general: 19
IDs solo en bloque financiero: 0


### 2.15 Análisis y caracterización de faltantes estructurales

Se identificó un subconjunto de 4.906 registros con ausencia simultánea de información en múltiples variables contractuales.

De estos registros, 4.887 presentan además ausencia completa de las variables financieras analizadas. La comparación mediante `id_contrato` confirmó que los 4.887 registros pertenecen al mismo subconjunto en ambos análisis, mientras que existen 19 registros adicionales que presentan faltantes estructurales pero conservan información financiera.

El análisis de `tipo_de_contrato` mostró que el 99,61 % de los registros con faltantes estructurales presentan esta variable como faltante, frente al 19,09 % observado en el conjunto completo.

Los registros con faltantes estructurales se encuentran distribuidos entre diferentes estados contractuales, por lo que no se consideran exclusivos de contratos en estado Borrador o Cancelado.

**Conclusión:** los faltantes presentan un patrón estructural y altamente concentrado en un subconjunto específico de registros. No se realizará imputación automática de estos valores, debido a que no existe evidencia suficiente para determinar los valores correctos. Los registros serán conservados y su cobertura será considerada en los análisis posteriores.